# Api to access sample inventory data.

In [ ]:
import os  # API keys are read from environment variables
# auto.dev api key
os.environ["AUTODEV_API_KEY"]


In [ ]:
import json

# Load VINs from JSON
with open("inventory_with_photos.json", "r") as f:
    data = json.load(f)

# Extract VINs
vins = [item["vin"] for item in data]

# Save VINs to a text file, one per line
with open("vins.txt", "w") as f:
    for vin in vins:
        f.write(f"{vin}\n")

print(f"✅ Saved {len(vins)} VINs to vins.txt")


In [ ]:
curl -X GET "https://api.auto.dev/photos/10CPACEK1RZ008314" \
  -H "Authorization: Bearer $AUTODEV_API_KEY" \
  -H "Content-Type: application/json"

In [ ]:
import requests
import json
import time

API_KEY = os.environ["AUTODEV_API_KEY"]
LISTINGS_URL = "https://api.auto.dev/listings"
PHOTOS_URL = "https://api.auto.dev/photos"

def get_photos(vin):
    """Fetch photos for a given VIN"""
    headers = {"Authorization": f"Bearer {API_KEY}"}
    url = f"{PHOTOS_URL}/{vin}"

    try:
        r = requests.get(url, headers=headers)
        if r.status_code == 200:
            data = r.json()
            return [p.get("url") for p in data.get("photos", [])]
    except Exception as e:
        print(f"Error fetching photos for {vin}: {e}")
    return []

def fetch_bulk_listings(limit=500, output_file="inventory_with_photos.json"):
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
    listings_per_page = 50
    total_pages = (limit + listings_per_page - 1) // listings_per_page
    all_listings = []

    for page in range(1, total_pages + 1):
        params = {"page": page, "limit": min(listings_per_page, limit - len(all_listings))}
        resp = requests.get(LISTINGS_URL, headers=headers, params=params)

        if resp.status_code != 200:
            print(f"Error {resp.status_code}: {resp.text}")
            break

        data = resp.json().get("data", [])
        for listing in data:
            vin = listing.get("vehicle", {}).get("vin")
            item = {
                "vin": vin,
                "make": listing.get("vehicle", {}).get("make"),
                "model": listing.get("vehicle", {}).get("model"),
                "year": listing.get("vehicle", {}).get("year"),
                "price": listing.get("retailListing", {}).get("price"),
                "mileage": listing.get("vehicle", {}).get("mileage"),
                "photos": get_photos(vin) if vin else []
            }
            all_listings.append(item)
            print(f"Fetched photos for {vin} ({len(all_listings)}/{limit})")

            if len(all_listings) >= limit:
                break

        time.sleep(1)  # small delay to avoid rate limits

    with open(output_file, "w") as f:
        json.dump(all_listings, f, indent=2)
    print(f"✅ Saved {len(all_listings)} listings with photos to {output_file}")

# Run
fetch_bulk_listings(limit=500)


# Use the Google Api key with tier 1

In [ ]:
import os

GOOGLE_API_KEY= os.environ["GOOGLE_API_KEY"]



os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [ ]:
Jina_api_key = os.environ["JINA_API_KEY"]

In [ ]:
PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
#This is for second email account

In [ ]:
from langchain import hub
from langchain import PromptTemplate
from langchain.docstore.document import Document
from langchain.document_loaders import WebBaseLoader
from langchain.schema import StrOutputParser
from langchain.schema.prompt_template import format_document
from langchain.schema.runnable import RunnablePassthrough
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from langchain_community.llms import Ollama
import streamlit as st

from pinecone import PodSpec

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

gemini_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-exp-03-07")

In [ ]:
from pinecone import Pinecone as pc

PINECONE_HOST="https://your-index.svc.your-env.pinecone.io"
PINECONE_INDEX="classicalbooks"

# Initialize Pinecone client
pine_client = pc(api_key=PINECONE_API_KEY)
index_name = PINECONE_INDEX
index = pine_client.Index(index_name, PINECONE_HOST)
vectorstore = PineconeVectorStore(
    index=index,
    embedding=gemini_embeddings,
    namespace="Default"
)


1) add documents using txt file or inventory api or whatever you have.
2) Split and chunk the content properly


In [ ]:
import json
from langchain.docstore.document import Document

docs = []

# Load inventory data from JSON
with open("inventory_with_photos.json", "r", encoding="utf-8") as f:
    inventory_data = json.load(f)

# Create documents from JSON data
for item in inventory_data:
    # Build structured content
    content_parts = [
        f"VIN: {item.get('vin', 'N/A')}",
        f"Make: {item.get('make', 'N/A')}",
        f"Model: {item.get('model', 'N/A')}",
        f"Year: {item.get('year', 'N/A')}",
        f"Price: ${item.get('price', 0):,}" if item.get('price') else "Price: Not available",
    ]
    
    # Add mileage if available
    if item.get('mileage') is not None:
        content_parts.append(f"Mileage: {item.get('mileage'):,} miles")
    
    content = "\n".join(content_parts)
    
    # Metadata: include only key fields for filtering
    metadata = {
        "vin": item.get("vin", ""),
        "make": item.get("make", ""),
        "model": item.get("model", "")
    }
    
    # Add as a Document
    docs.append(Document(page_content=content, metadata=metadata))

print(f"✅ Created {len(docs)} documents from inventory_with_photos.json")



In [ ]:

# Or in batches (for large files)
batch_size = 50
for i in range(0, len(docs), batch_size):
    batch = docs[i:i+batch_size]
    vectorstore.add_documents(batch)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Search should output structured content which will be shown on website. 

In [ ]:
docs = retriever.invoke("show me Acura under 50000")
if docs:
    for i, doc in enumerate(docs, 1):
        print(f"📄 Document {i}:")
        print("-" * 60)
        print(doc.page_content)
        #print("\n🔖 Metadata:")
        #for key, value in doc.metadata.items():
            #print(f"   {key}: {value}")
        #print()  # Blank line for readability
else:
    print("❌ No documents found above any threshold.")

